# 01 — OSM Preprocessing Pipeline

Downloads ward-level OSM data via the Overpass API using BBMP ward boundaries,
then compiles each into a SUMO network using `netconvert`.

**Outputs per ward:**
- `maps/raw_osm/ward_XXX.osm` — raw OpenStreetMap XML
- `maps/processed/ward_XXX/ward.net.xml` — SUMO network
- `maps/processed/ward_XXX/metadata.json` — structural metadata
- `maps/processed/ward_XXX/boundaries.json` — ingress/egress boundary edges

In [ ]:
import sys, os
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path(os.getcwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## 1. Load Ward Registry

In [ ]:
import json
import pandas as pd

from src.preprocessing.osm_fetcher import load_ward_registry

registry = load_ward_registry(PROJECT_ROOT)
wards = registry["wards"]

print(f"Total wards registered: {len(wards)}")

# Display ward summary table
rows = []
for wid, meta in wards.items():
    rows.append({
        "ward_id": wid,
        "label": meta["label"],
        "zone_type": meta["zone_type"],
        "parent_area": meta["parent_area"],
        "parent_region": meta["parent_region"],
        "congestion_prior": meta["congestion_prior"],
        "hospital_sensitive": meta["hospital_sensitive"],
    })

df_wards = pd.DataFrame(rows)
df_wards

## 2. Download OSM Data via Overpass API

This will download `.osm` files for all wards into `maps/raw_osm/`.
Files already downloaded will be skipped (set `force=True` to re-download).

In [ ]:
from src.preprocessing.osm_fetcher import fetch_all_wards, validate_osm_file

# Download all wards (set force=True to re-download)
results = fetch_all_wards(PROJECT_ROOT, force=False)

# Display download results
df_download = pd.DataFrame(results)
print(f"\nDownload summary:")
print(f"  OK:      {(df_download['status'] == 'ok').sum()}")
print(f"  Skipped: {(df_download['status'] == 'skipped').sum()}")
print(f"  Errors:  {df_download['status'].str.startswith('error').sum()}")
df_download

## 3. Validate Downloaded OSM Files

In [ ]:
raw_dir = PROJECT_ROOT / "maps" / "raw_osm"
osm_files = sorted(raw_dir.glob("ward_*.osm"))

print(f"OSM files found: {len(osm_files)}")
for f in osm_files:
    valid = validate_osm_file(f)
    size_kb = f.stat().st_size / 1024
    status = "✅" if valid else "❌"
    print(f"  {status} {f.name} ({size_kb:.1f} KB)")

## 4. Compile SUMO Networks (netconvert)

For each ward OSM file, runs `netconvert` to produce:
- `ward.net.xml` — SUMO network
- `metadata.json` — edge/junction statistics
- `boundaries.json` — ingress/egress boundary edges

In [ ]:
from src.preprocessing.ward_processor import process_all_wards

processing_results = process_all_wards(PROJECT_ROOT)

# Display results
for r in processing_results:
    status = "✅" if r["status"] == "ok" else "❌"
    print(f"  {status} {r['ward_id']}: {r['status']}")

## 5. Inspect Processed Ward Networks

In [ ]:
processed_dir = PROJECT_ROOT / "maps" / "processed"
ward_dirs = sorted(processed_dir.iterdir()) if processed_dir.exists() else []

rows = []
for wd in ward_dirs:
    if not wd.is_dir():
        continue
    meta_path = wd / "metadata.json"
    bound_path = wd / "boundaries.json"
    net_path = wd / "ward.net.xml"
    
    row = {"ward_id": wd.name, "net_exists": net_path.exists()}
    
    if meta_path.exists():
        with meta_path.open() as f:
            meta = json.load(f)
        net_info = meta.get("network", {})
        row["zone_type"] = meta.get("zone_type", "?")
        row["edges"] = net_info.get("edge_count", "?")
        row["junctions"] = net_info.get("junction_count", "?")
        row["signals"] = net_info.get("traffic_signal_count", "?")
    
    if bound_path.exists():
        with bound_path.open() as f:
            bounds = json.load(f)
        row["ingress"] = bounds.get("ingress_count", 0)
        row["egress"] = bounds.get("egress_count", 0)
    
    rows.append(row)

if rows:
    df_processed = pd.DataFrame(rows)
    display(df_processed)
else:
    print("No processed wards found yet. Run step 4 first.")

## 6. Quick Sanity Check — Load a Ward Network in SUMO

Pick the first processed ward and verify it loads without errors.

In [ ]:
if ward_dirs:
    test_ward = ward_dirs[0].name
    net_file = processed_dir / test_ward / "ward.net.xml"
    if net_file.exists():
        print(f"✅ Network ready: {net_file}")
        print(f"   Size: {net_file.stat().st_size / 1024:.1f} KB")
        print(f"\nTo view in SUMO GUI, run:")
        print(f'   sumo-gui -n "{net_file}"')
    else:
        print(f"❌ Network not found for {test_ward}")
else:
    print("No processed wards available.")